In [1]:
corpus = [
    # --- account cluster ---
    "To reset a franchise admin password open the admin console and choose account recovery",   # 0
    "If your account is locked after several failed login attempts contact your regional manager", # 1
    # --- refund / money-back: PARAPHRASE targets, note: no words 'money' or 'back' ---
    "Refunds are processed within seven business days to the original payment method",           # 2
    "Reimbursement for approved franchise expenses appears on your next monthly statement",       # 3
    # --- exact code / jargon: BM25 targets ---
    "Error code E-4021 indicates a payment gateway timeout during checkout",                      # 4
    "The SKU checker rejects any product barcode that fails the GTIN-13 checksum",                # 5
    # --- k1 saturation trap: doc 6 repeats 'inventory' 5x, doc 7 once ---
    "Inventory inventory inventory sync keeps store inventory levels accurate across each inventory node", # 6
    "The point of sale terminal syncs inventory every fifteen minutes",                           # 7
    # --- b length trap: same term counts, wildly different length ---
    "Royalty fees are due monthly",                                                               # 8
    "Royalty fees representing an agreed percentage of gross sales are invoiced and franchisees "
    "should review the detailed breakdown in the financial portal before each payment deadline "
    "to avoid late penalties and reconciliation issues across all their individual store accounts", # 9
    # --- distractors ---
    "The onboarding wizard guides new franchisees through initial store setup",                   # 10
    "Marketing materials can be requested through the brand asset portal",                        # 11
]

In [2]:
import math
from collections import Counter

def tokenize(text):
    return text.lower().split()

class BM25:
    def __init__(self, corpus, k1 = 1.5, b = 0.75):
        self.k1, self.b = k1, b
        self.docs = [ tokenize(d) for d in corpus]
        self.N = len(self.docs)
        self.doc_len = [ len(d) for d in self.docs]
        self.avgdl = sum(self.doc_len) / self.N
        self.df = Counter()
        for d in self.docs:
            for term in set(d):
                self.df[term] += 1
        self.idf = { t: math.log(1 + (self.N - f + 0.5) / (f + 0.5)) for t, f in self.df.items()}

    def score(self, query, i):
        doc = self.docs[i]
        freqs = Counter(doc)
        dl = self.doc_len[i]
        s = 0.0
        for t in tokenize(query):
            if t not in self.idf:
                continue
            f = freqs[t]
            if f == 0:
                continue
            num = f * (self.k1 + 1)
            den = f + self.k1 * ( 1 - self.b + self.b * dl/self.avgdl)
            s += self.idf[t] * num / den
        return s

    def search(self, query, top_k=5):
        ranked = sorted(((i, self.score(query, i )) for i in range(self.N)), key = lambda x: -x[1])
        return ranked[:top_k]


In [3]:
import random
random.seed(0)
vocab = "staff schedule shift report dashboard notification setting profile district audit compliance training module ticket".split()
filler = [" ".join(random.choices(vocab, k=random.randint(8, 30))) for _ in range(188)]
corpus_full = corpus + filler   # 12 planted + 188 filler = 200

bm = BM25(corpus_full)
print(bm.search("reset admin password"))   # sanity check → doc 0 should top

[(0, 18.472844099785167), (1, 0.0), (2, 0.0), (3, 0.0), (4, 0.0)]


In [4]:
for k1 in [0.0, 0.5, 1.5, 5.0, 20.0]:
    bm = BM25(corpus_full, k1=k1, b=0.75)
    hits = [(i, round(s, 3)) for i, s in bm.search("inventory sync", top_k=3)]
    print(f"k1={k1:<5} {hits}")

k1=0.0   [(6, 9.285), (7, 4.387), (0, 0.0)]
k1=0.5   [(6, 11.372), (7, 4.942), (0, 0.0)]
k1=1.5   [(6, 14.488), (7, 5.498), (0, 0.0)]
k1=5.0   [(6, 20.684), (7, 6.099), (0, 0.0)]
k1=20.0  [(6, 28.353), (7, 6.46), (0, 0.0)]


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")   # small, fast, CPU-fine
emb = model.encode(corpus_full, normalize_embeddings=True)

def dense_search(query, top_k=5):
    q = model.encode([query], normalize_embeddings=True)[0]
    sims = emb @ q                          # cosine, since normalized
    idx = np.argsort(-sims)[:top_k]
    return [(int(i), round(float(sims[i]), 3)) for i in idx]

c:\Users\durga\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
q = "E-4021"
print("BM25 :", BM25(corpus_full).search(q, top_k=3))
print("dense:", dense_search(q, top_k=3))

BM25 : [(4, 6.138677728593486), (0, 0.0), (1, 0.0)]
dense: [(4, 0.473), (130, 0.247), (193, 0.231)]


In [7]:
q = "how can I get my money back"
print("BM25 :", BM25(corpus_full).search(q, top_k=3))
print("dense:", dense_search(q, top_k=3))

BM25 : [(11, 6.138677728593486), (0, 0.0), (1, 0.0)]
dense: [(2, 0.46), (1, 0.341), (0, 0.289)]


BM25 - Best match 25, where higher the score means best match. BM25 is adding 2 knobs on top of TF IDF. TF IDF is the traditional or first techniques for Sparse Retreival. TF - Term Frequency in the document, IDF : Inverse Document Frequency. Basically means TF - How many times the doc has the term. IDF - How many documents doesn't have the term, its like how rare is the term compared to how many docs its present over all documents. And Bm25 adds K1 and b knobs to regular TF IDF, where k1 is the knob for term saturation. meaning k1=0 means frequency of term doesn't matter in a doc, score per occurance not for frequency, increasing k1 means frequency will start effecting the score as in the term is more frequent in a doc it gets high score. where as b is the length normalilzation, means how much effect length of the document has in the scoring. b=0 means no effect over scoring. A sentence with 6 words containing term and sentence with 50 words containing term gets same score. but with higher b, a short sentence or doc that matches the term gets higher score than a large doc containig the term. 

stemmers and analyzers make sure that words match such as sync and syncs / syncing will match, it is taken care by stemmers and analyzers.


TF — Term Frequency — is about ONE document. How many times does the term appear in this specific doc? "inventory" appears 5× in doc 6 → high TF for doc 6. It's a within-document count. Nothing to do with how many docs contain it.

IDF — Inverse Document Frequency — is about the WHOLE corpus. How rare is the term across all docs? It's built from document frequency (DF) = how many docs contain the term. Then IDF inverts that: rare term (low DF) → high IDF; common term (high DF) → low IDF. "the" is in every doc → DF huge → IDF ≈ 0 → worthless. "E-4021" is in 1 doc → DF tiny → IDF huge → decisive.